# Test Fine-tuning Results
Test the fine-tuned LIANet Creoss region performance to the local model perfomance

In [17]:

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "2"
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from helpers import load_model_class
import glob
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from rasterio.windows import Window
# Add src to path
sys.path.insert(0, '/home/user/src')
import rasterio as rio
from datasets import BurnScars
from models.models_finetune import UNet, MicroUNet
from settings import *



from torchmetrics import MetricCollection

from metrics import multiclass_segmentation_metrics
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [18]:
import json
from models.models_finetune import DownstreamModel
from models.LIANet import LIANetLight
import omegaconf, hydra

# pretrained_model_path = "/home/user/results_shared/fourier_learned_HLS_2regions/2026-04-22_15-34-35"
pretrained_model_path = {"BurnScars_joint_T11SMT": "/home/user/results_shared/fourier_learned_HLS_2regions/2026-04-22_15-34-35",
                         "BurnScars_joint_T16REV": "/home/user/results_shared/fourier_learned_HLS_2regions/2026-04-22_15-34-35",
                         "BurnScars_local_T11SMT": "/home/user/results_shared/fourier_learned_HLS_T11SMT/2026-04-30_20-30-35",
                         "BurnScars_local_T16REV": "/home/user/results_shared/fourier_learned_HLS_T16REV/2026-04-28_09-13-10"}



def load_model(CKPT_PATH, other_task, model_type):
    model_finetune = load_model_class(
        other_task, 
        model_type, 
        MODEL_PATH= pretrained_model_path[other_task],
        NUM_CLASSES=num_classes[other_task],
        ACTIVATION_FUNCTION=activation_functions[other_task]
    )
    # model_finetune = DownstreamModel(
    #     model_path=pretrained_model_path,
    #     checkpoint_path_relative="model_checkpoints/latest_validation_checkpoint.pt",
    #     adaption_strategy="replace_final_block",
    #     num_classes=num_classes[other_task],
    #     activation="none"
    # )

    checkpoint = torch.load(CKPT_PATH, map_location=device)
    state_dict = checkpoint["model_state_dict"]
    if state_dict and all(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    model_finetune.load_state_dict(state_dict, strict=True)
    model_finetune = model_finetune.to(device)
    model_finetune.eval()
    # print("✓ Model loaded successfully")
    return model_finetune

In [19]:
import os
import torch
import pandas as pd

from tqdm import tqdm
from torchmetrics import MetricCollection
from metrics import multiclass_segmentation_metrics


model_type_list = ["replace_final_block"]

for model_type in model_type_list:

    if model_type == "replace_final_block":
        all_tasks_list = {
            "BurnScars_joint_T11SMT": ["BurnScars_joint_T16REV"],
            "BurnScars_joint_T16REV": ["BurnScars_joint_T11SMT"],
            # "BurnScars_local_T11SMT": ["BurnScars_local_T11SMT"],
            # "BurnScars_local_T16REV": ["BurnScars_local_T16REV"],
        }
    elif model_type == "unet" or model_type == "micro_unet":
        all_tasks_list = {
            "BurnScars_local_T11SMT": ["BurnScars_local_T16REV"],
            "BurnScars_local_T16REV": ["BurnScars_local_T11SMT"],
        }

    BATCH_SIZE = 16
    NUM_WORKERS = 8

    results_rows = []


    for Target_region in all_tasks_list:
        application_name = Target_region.split("_")[0]
        other_tasks = all_tasks_list[Target_region]

        val_dataset = BurnScars(
            top_dir=TOP_DIR[Target_region],
            s2_tiles=s2_tiles[Target_region],
            labels=labels[Target_region],
            train_val_key="val",
        )

        dataloader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            drop_last=False,
        )

        for source_region in other_tasks:
            startername = "LIANet" if "replace_final_block" in model_type else "microunet" if "micro_unet" in model_type else "unet"
            model_dir = glob.glob(f"/home/user/results_local/finetuning_results_BurnScars_valVisualization/{source_region}/{startername}*")
            # model_dir = (
            #     f"/home/user/results_local/finetuning_results_BurnScars/"
            #     f"{source_region}/LIANet_lr0.0003_batchsize32_nonburned"
            # )

            for run_name in sorted(os.listdir(model_dir[0])):
                ckpt_path = os.path.join(model_dir[0], run_name, "last.pt")

                if not os.path.exists(ckpt_path):
                    continue

                model = load_model(ckpt_path, source_region, model_type)
                model.eval()

                list_of_metrics, _ = multiclass_segmentation_metrics(
                    num_classes=num_classes[Target_region],
                )

                metrictracker = MetricCollection(list_of_metrics).to(device)

                with torch.no_grad():
                    for batch in tqdm(
                        dataloader,
                        desc=f"{Target_region} | {source_region} | {run_name}",
                        leave=False,
                    ):
                        x = batch["x_s2"].to(device)
                        y = batch["y_s2"].to(device)
                        label = batch["label"].to(device)
                        delta_days = batch["delta_days"].to(device)
                        target_tile = Target_region.split("_")[-1]
                        s2data  = batch["s2data"].to(device)
                        if "local" in Target_region:
                            region_idx = 0
                        elif "BFP" in Target_region:
                            region_idx = 1 if target_tile == "T32ULU" else 2 if target_tile == "T31TFM" else None
                        elif "joint" and "PASTIS" in Target_region:
                            region_indx = 0 if target_tile == "T31TFJ" else 1 if target_tile == "T32ULU" else 2 if target_tile == "T31TFM" else 3
                        elif "joint" and "BurnScars" in Target_region:
                            region_idx = 0 if target_tile == "T11SMT" else 1 if target_tile == "T16REV" else None
                        if "replace_final_block" in model_type:
                            _, pred = model(
                                delta_days,
                                x,
                                y,
                                torch.tensor([region_idx], device=device),
                            )
                        elif "micro_unet" in model_type or "unet" in model_type:
                            pred = model(s2data)

                        pred = getattr(pred, "output", pred)

                        if pred.dim() == 3:
                            pred = pred.unsqueeze(1)

                        metrictracker.update(pred.squeeze(1), label)

                results = metrictracker.compute()

                row = {
                    "model_type": model_type,
                    "target_region": Target_region,
                    "source_region": source_region,
                    "seed_or_run": run_name,
                    "checkpoint_path": ckpt_path,
                }

                for metric_name, metric_value in results.items():
                    row[metric_name] = float(metric_value.detach().cpu())

                    results_rows.append(row)

                del model
                del metrictracker
                torch.cuda.empty_cache()

        del dataloader
        del val_dataset
        torch.cuda.empty_cache()


    df_results = pd.DataFrame(results_rows)

    df_results.to_csv(
        "{}_BurnScars_results_all_target_regions.csv".format(model_type),
        index=False,
    )


Building val image label pairs: 100%|██████████| 2/2 [00:00<00:00,  8.52it/s]


Found 17 samples for val with >0% burned area
Average burned pixel count across samples: 6307.941176470588


Building val image label pairs: 100%|██████████| 4/4 [00:00<00:00,  8.37it/s]                                       


Found 48 samples for val with >0% burned area
Average burned pixel count across samples: 5978.8125


In [20]:
# smt_train_dataset = BurnScars(
#             top_dir="/home/user/data_shared",
#             s2_tiles="T11SMT",
#             labels="masks/BurnScars",
#             train_val_key="train",
#         )
# smt_val_dataset = BurnScars(
#             top_dir="/home/user/data_shared",
#             s2_tiles="T11SMT",
#             labels="masks/BurnScars",
#             train_val_key="val",
#         )
# rev_val_dataset = BurnScars(
#             top_dir="/home/user/data_shared",
#             s2_tiles="T16REV",
#             labels="masks/BurnScars",
#             train_val_key="val",
#         )
# rev_train_dataset = BurnScars(
#             top_dir="/home/user/data_shared",
#             s2_tiles="T16REV",
#             labels="masks/BurnScars",
#             train_val_key="train",
#         )

In [21]:
import pandas as pd

model_type_list = ["replace_final_block"]

avg_results = {}

for model_type in model_type_list:

    csv_path = f"{model_type}_BurnScars_results_all_target_regions.csv"

    print(f"\n========== {model_type} ==========")

    df_results = pd.read_csv(csv_path)

    df_results["training_type"] = (
        df_results["target_region"]
        .str.extract(r"_(joint|local)_")
    )

    metadata_cols = [
        "model_type",
        "target_region",
        "source_region",
        "seed_or_run",
        "checkpoint_path",
        "training_type",
    ]

    metric_cols = [
        col for col in df_results.select_dtypes(include="number").columns
        if col not in metadata_cols
    ]

    df_seed_avg = (
        df_results
        .groupby(
            ["training_type", "target_region", "source_region"],
            as_index=False
        )[metric_cols]
        .mean()
    )

    df_joint_local_avg = (
        df_seed_avg
        .groupby("training_type", as_index=False)[metric_cols]
        .mean()
    )

    avg_results[model_type] = df_joint_local_avg

    print(df_joint_local_avg)


========== replace_final_block ==========
  training_type  accuracy_macro  accuracy_micro  f1_macro  f1_micro  \
0         joint        0.752149        0.774011  0.747323  0.774011   

   jaccard_macro  jaccard_micro  precision_macro  precision_micro  \
0        0.60569       0.633068         0.792873         0.774011   

   recall_macro  recall_micro  
0      0.752149      0.774011  


In [13]:
import json
from models.models_finetune import DownstreamModel
from models.LIANet import LIANetLight
import omegaconf, hydra

# pretrained_model_path = "/home/user/results_shared/fourier_learned_4regions/2026-03-18_19-06-01"

pretrained_model_path_list = {
    "T11SMT": "/home/user/results_shared/fourier_learned_HLS_T11SMT/2026-04-30_20-30-35",
    "T16REV": "/home/user/results_shared/fourier_learned_HLS_T16REV/2026-04-28_09-13-10",
}

def load_model(CKPT_PATH, other_task):

    pretrained_model_path = pretrained_model_path_list[other_task.split("_")[-1]]
    model_finetune = DownstreamModel(
        model_path=pretrained_model_path,
        checkpoint_path_relative="model_checkpoints/latest_validation_checkpoint.pt",
        adaption_strategy="replace_final_block",
        num_classes=num_classes[other_task],
        activation="none"
    )

    checkpoint = torch.load(CKPT_PATH, map_location=device)
    state_dict = checkpoint["model_state_dict"]
    if state_dict and all(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    model_finetune.load_state_dict(state_dict, strict=True)
    model_finetune = model_finetune.to(device)
    model_finetune.eval()
    # print("✓ Model loaded successfully")
    return model_finetune

In [15]:
import os
import torch
import pandas as pd

from tqdm import tqdm
from torchmetrics import MetricCollection
from metrics import multiclass_segmentation_metrics


all_tasks_list = {
    "BurnScars_local_T11SMT",
    "BurnScars_local_T16REV",
}

BATCH_SIZE = 16
NUM_WORKERS = 8

results_rows = []


for Target_region in all_tasks_list:
    source_region = Target_region

    val_dataset = BurnScars(
        top_dir=TOP_DIR[Target_region],
        s2_tiles=s2_tiles[Target_region],
        labels=labels[Target_region],
        train_val_key="val",
    )

    dataloader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        drop_last=False,
    )

    model_dir = (
        f"/home/user/results_local/finetuning_results_BurnScars/"
        f"{source_region}/LIANet_lr0.0003_batchsize32_nonburned"
    )

    if not os.path.exists(model_dir):
        print(f"Missing folder: {model_dir}")
        continue

    for run_name in sorted(os.listdir(model_dir)):
        ckpt_path = os.path.join(model_dir, run_name, "last.pt")

        if not os.path.exists(ckpt_path):
            continue

        model = load_model(ckpt_path, source_region)
        model.eval()

        list_of_metrics, _ = multiclass_segmentation_metrics(
            num_classes=num_classes[Target_region],
        )

        metrictracker = MetricCollection(list_of_metrics).to(device)

        with torch.no_grad():
            for batch in tqdm(
                dataloader,
                desc=f"{Target_region} | {run_name}",
                leave=False,
            ):
                x = batch["x_s2"].to(device)
                y = batch["y_s2"].to(device)
                label = batch["label"].to(device)
                delta_days = batch["delta_days"].to(device)

                _, pred = model(
                    delta_days,
                    x,
                    y,
                    torch.tensor([0], device=device),
                )

                pred = getattr(pred, "output", pred)

                if pred.dim() == 3:
                    pred = pred.unsqueeze(1)

                metrictracker.update(pred, label)

        results = metrictracker.compute()

        row = {
            "target_region": Target_region,
            "source_region": source_region,
            "seed_or_run": run_name,
            "checkpoint_path": ckpt_path,
        }

        for metric_name, metric_value in results.items():
            row[metric_name] = float(metric_value.detach().cpu())

        results_rows.append(row)

        del model
        del metrictracker
        torch.cuda.empty_cache()

    del dataloader
    del val_dataset
    torch.cuda.empty_cache()


df_local_results = pd.DataFrame(results_rows)

df_local_results.to_csv(
    "BurnScars_local_baseline_results_all_regions.csv",
    index=False,
)

df_local_results

Building val image label pairs: 100%|██████████| 2/2 [00:00<00:00,  8.88it/s]


Found 17 samples for val with >0% burned area
Average burned pixel count across samples: 6307.941176470588


Building val image label pairs: 100%|██████████| 4/4 [00:00<00:00,  8.47it/s]              


Found 48 samples for val with >0% burned area
Average burned pixel count across samples: 5978.8125


,target_region,source_region,seed_or_run,checkpoint_path,accuracy_macro,accuracy_micro,f1_macro,f1_micro,jaccard_macro,jaccard_micro,precision_macro,precision_micro,recall_macro,recall_micro
0,BurnScars_local_T11SMT,BurnScars_local_T11SMT,2026-05-10_14-39-21,/home/user/results_local/finetuning_results_Bu...,0.934058,0.936541,0.933138,0.936541,0.875031,0.880656,0.932259,0.936541,0.934058,0.936541
1,BurnScars_local_T11SMT,BurnScars_local_T11SMT,2026-05-10_14-41-08,/home/user/results_local/finetuning_results_Bu...,0.936342,0.938613,0.935333,0.938613,0.878873,0.884327,0.934372,0.938613,0.936342,0.938613
2,BurnScars_local_T11SMT,BurnScars_local_T11SMT,2026-05-10_14-42-53,/home/user/results_local/finetuning_results_Bu...,0.934590,0.938911,0.935381,0.938911,0.878984,0.884856,0.936202,0.938911,0.934590,0.938911
3,BurnScars_local_T11SMT,BurnScars_local_T11SMT,2026-05-10_14-44-41,/home/user/results_local/finetuning_results_Bu...,0.934991,0.935791,0.932571,0.935791,0.874019,0.879330,0.930433,0.935791,0.934991,0.935791
4,BurnScars_local_T11SMT,BurnScars_local_T11SMT,2026-05-10_14-46-27,/home/user/results_local/finetuning_results_Bu...,0.934785,0.935507,0.932286,0.935507,0.873518,0.878829,0.930087,0.935507,0.934785,0.935507
5,BurnScars_local_T16REV,BurnScars_local_T16REV,2026-05-10_14-39-16,/home/user/results_local/finetuning_results_Bu...,0.836293,0.835625,0.826985,0.835625,0.706862,0.717659,0.821802,0.835625,0.836293,0.835625
6,BurnScars_local_T16REV,BurnScars_local_T16REV,2026-05-10_14-40-58,/home/user/results_local/finetuning_results_Bu...,0.818851,0.812342,0.804755,0.812342,0.675034,0.683987,0.800202,0.812342,0.818851,0.812342
7,BurnScars_local_T16REV,BurnScars_local_T16REV,2026-05-10_14-42-43,/home/user/results_local/finetuning_results_Bu...,0.834484,0.831497,0.823379,0.831497,0.701545,0.711592,0.818073,0.831497,0.834484,0.831497
8,BurnScars_local_T16REV,BurnScars_local_T16REV,2026-05-10_14-44-27,/home/user/results_local/finetuning_results_Bu...,0.837581,0.830827,0.823684,0.830827,0.701772,0.710611,0.818526,0.830827,0.837581,0.830827
9,BurnScars_local_T16REV,BurnScars_local_T16REV,2026-05-10_14-46-08,/home/user/results_local/finetuning_results_Bu...,0.834695,0.834106,0.825398,0.834106,0.704583,0.715422,0.820241,0.834106,0.834695,0.834106


In [16]:
metadata_cols = [
    "target_region",
    "source_region",
    "seed_or_run",
    "checkpoint_path",
]

metric_cols = [
    col for col in df_local_results.columns
    if col not in metadata_cols
]

# Average over seeds/runs
df_local_seed_avg = (
    df_local_results
    .groupby(["target_region", "source_region"], as_index=False)[metric_cols]
    .mean()
)

# Final average over all local baselines
df_local_final_avg = (
    df_local_seed_avg[metric_cols]
    .mean()
    .to_frame()
    .T
)

df_local_seed_avg.to_csv(
    "BurnScars_local_baseline_avg_per_region.csv",
    index=False,
)

df_local_final_avg.to_csv(
    "BurnScars_local_baseline_final_avg_all_regions.csv",
    index=False,
)

df_local_seed_avg, df_local_final_avg

(            target_region           source_region  accuracy_macro  \
 0  BurnScars_local_T11SMT  BurnScars_local_T11SMT        0.934954   
 1  BurnScars_local_T16REV  BurnScars_local_T16REV        0.832381   
 
    accuracy_micro  f1_macro  f1_micro  jaccard_macro  jaccard_micro  \
 0        0.937073  0.933742  0.937073       0.876085       0.881600   
 1        0.828880  0.820840  0.828880       0.697959       0.707854   
 
    precision_macro  precision_micro  recall_macro  recall_micro  
 0         0.932671         0.937073      0.934954      0.937073  
 1         0.815769         0.828880      0.832381      0.828880  ,
    accuracy_macro  accuracy_micro  f1_macro  f1_micro  jaccard_macro  \
 0        0.883667        0.882976  0.877291  0.882976       0.787022   
 
    jaccard_micro  precision_macro  precision_micro  recall_macro  recall_micro  
 0       0.794727          0.87422         0.882976      0.883667      0.882976  )